# Home Assignment
## Two molecules a graph model cannot tell apart



---

### The chemistry

Compare the carbon skeletons of two hydrocarbons, hydrogens suppressed throughout.

```
        A: cyclohexane skeleton              B: two cyclopropane skeletons

               0                                   0            3
            /     \                               / \          / \
           1       5                             1---2        4---5
           |       |
           2       4
            \     /
               3
```

Both graphs have **six carbon atoms** and **six carbon-carbon bonds**, and every atom
carries the same initial label, `"C"`. Chemically they could hardly be more different:
one is a strain-free chair, the other carries roughly 115 kJ/mol of ring strain **per ring**.

Your task is to establish exactly what a standard message-passing model can and cannot
perceive here, and then to fix it.



In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

def show(name, A):
    """Print an adjacency matrix with its degree column."""
    print(f'{name}   (degrees on the right)')
    for i, row in enumerate(A.astype(int)):
        print('  ' + ' '.join(str(v) for v in row) + f'   | d_{i} = {int(row.sum())}')
    print(f'  bonds = {int(A.sum() // 2)},  atoms = {A.shape[0]}')
    print()


---
## Task 1. Write both adjacency matrices  &nbsp;&nbsp;

Fill in `A_A` and `A_B` using the atom labels in the diagram above.

Reminders:
- The adjacency matrix is $A_{ij}=1$ when atoms $i$ and $j$ are bonded, and $0$ otherwise.
- A molecular graph is undirected, so $A$ must be **symmetric**.
- There are no self-bonds, so the diagonal is zero.
- In **B** the two rings are *not* connected to each other. Atoms 0,1,2 form one ring
  and atoms 3,4,5 the other.


In [2]:
# TODO: replace the zeros with the correct entries.
# Ring A: 0-1, 1-2, 2-3, 3-4, 4-5, 5-0

A_A = np.zeros((6, 6))
A_A[0, 1] = A_A[1, 0] = 1.0        # <- one bond done for you as a pattern; uncomment and continue
A_A[1, 2] = A_A[2, 1] = 1.0
A_A[2, 3] = A_A[3, 2] = 1.0
A_A[3, 4] = A_A[4, 3] = 1.0
A_A[4, 5] = A_A[5, 4] = 1.0
A_A[5, 0] = A_A[0, 5] = 1.0


# TODO: Ring B: 0-1, 1-2, 2-0  and  3-4, 4-5, 5-3

A_B = np.zeros((6, 6))
A_B = np.zeros((6, 6))
A_B[0, 1] = A_B[1, 0] = 1.0
A_B[1, 2] = A_B[2, 1] = 1.0
A_B[2, 0] = A_B[0, 2] = 1.0
A_B[3, 4] = A_B[4, 3] = 1.0
A_B[4, 5] = A_B[5, 4] = 1.0
A_B[5, 3] = A_B[3, 5] = 1.0

show('A  (cyclohexane skeleton)', A_A)
show('B  (two cyclopropane skeletons)', A_B)


A  (cyclohexane skeleton)   (degrees on the right)
  0 1 0 0 0 1   | d_0 = 2
  1 0 1 0 0 0   | d_1 = 2
  0 1 0 1 0 0   | d_2 = 2
  0 0 1 0 1 0   | d_3 = 2
  0 0 0 1 0 1   | d_4 = 2
  1 0 0 0 1 0   | d_5 = 2
  bonds = 6,  atoms = 6

B  (two cyclopropane skeletons)   (degrees on the right)
  0 1 1 0 0 0   | d_0 = 2
  1 0 1 0 0 0   | d_1 = 2
  1 1 0 0 0 0   | d_2 = 2
  0 0 0 0 1 1   | d_3 = 2
  0 0 0 1 0 1   | d_4 = 2
  0 0 0 1 1 0   | d_5 = 2
  bonds = 6,  atoms = 6



In [3]:
# CHECK: structural properties only. These do not reveal the chemistry.
for name, A in [('A', A_A), ('B', A_B)]:
    assert A.shape == (6, 6),               f'{name}: must be 6x6'
    assert np.allclose(A, A.T),             f'{name}: must be symmetric'
    assert np.allclose(np.diag(A), 0),      f'{name}: diagonal must be zero'
    assert set(np.unique(A)) <= {0.0, 1.0}, f'{name}: entries must be 0 or 1'
    assert A.sum() // 2 == 6,               f'{name}: must have exactly 6 bonds'
assert not np.allclose(A_A, A_B), 'A and B must be different matrices'
print('Task 1 structural checks passed.')
print('Degree sequence A:', np.sort(A_A.sum(1)).astype(int))
print('Degree sequence B:', np.sort(A_B.sum(1)).astype(int))


Task 1 structural checks passed.
Degree sequence A: [2 2 2 2 2 2]
Degree sequence B: [2 2 2 2 2 2]


### YOUR ANSWER (Task 1)

State the degree of every atom in each graph, and say in one sentence what the degree
of a carbon atom means chemically.

*Write here:*



Graph A: All vertices exhibit a degree of 2 → [2, 2, 2, 2, 2, 2].

Graph B: All vertices exhibit a degree of 2 → [2, 2, 2, 2, 2, 2].

Chemically, the degree corresponds to the carbon atom's immediate bonding environment, quantifying the total number of adjacent atoms it shares a direct bond with

---
## Task 2. Colour refinement  &nbsp;&nbsp;

**Do this by hand first, on paper.** The code is to check your hand work, not to replace it.

The 1-WL refinement rule is

$$c^{(k+1)}_i \;=\; \mathrm{HASH}\Big(c^{(k)}_i,\ \{\!\{\,c^{(k)}_j : j \in \mathcal{N}(i)\,\}\!\}\Big)$$

where $\{\!\{\cdot\}\!\}$ is a **multiset** (repetition matters, order does not) and
$\mathrm{HASH}$ assigns a fresh integer to each distinct signature it has seen.

Two graphs are declared **distinguishable** if at some round their *multisets of colours*
differ.

Complete the function below.


In [4]:
def wl_round(A, colours):
    """One round of 1-WL colour refinement.

    A       : (n, n) adjacency matrix
    colours : list of n integers, the current colours
    returns : list of n integers, the refined colours
    """
    n = len(colours)
    signatures = []
    for i in range(n):
        # TODO: build the multiset of neighbour colours for atom i.
        #       Use sorted(...) to make the multiset order-independent,
        #       and tuple(...) so it can be used as a dictionary key.
        nb_cl = []
        for j in range(n):
            if A[i, j] == 1:
                nb_cl.append(colours[j])
        neighbour_colours = tuple(sorted(nb_cl))      # <- replace

        # TODO: the signature is the pair (own colour, neighbour multiset)
        signatures.append((colours[i], neighbour_colours))       # <- replace

    # relabel each distinct signature with a fresh integer (this part is done for you)
    table = {s: k for k, s in enumerate(sorted(set(signatures), key=str))}
    return [table[s] for s in signatures]


### Validate your implementation on a case with a known answer

Before trusting `wl_round` on A and B, test it on a pair where the answer is already known:
the carbon skeletons of **n-butane** (a chain) and **isobutane** (a central carbon with
three neighbours).

These two *are* distinguishable, and refinement should separate them after one round,
because their degree sequences differ.


In [5]:
# CHECK: a validation case with a known outcome. Do not edit.
A_nbutane  = np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]], float)
A_isobutane = np.array([[0,1,1,1],[1,0,0,0],[1,0,0,0],[1,0,0,0]], float)

c1 = wl_round(A_nbutane,  [0, 0, 0, 0])
c2 = wl_round(A_isobutane, [0, 0, 0, 0])
print('n-butane  colours after 1 round:', sorted(c1))
print('isobutane colours after 1 round:', sorted(c2))
assert sorted(c1) != sorted(c2), (
    'Your wl_round does not separate n-butane from isobutane, but it should. '
    'Check that you are using a MULTISET of neighbour colours, not a set.')
print('\nValidation passed: wl_round behaves correctly on a known case.')


n-butane  colours after 1 round: [0, 0, 1, 1]
isobutane colours after 1 round: [0, 1, 1, 1]

Validation passed: wl_round behaves correctly on a known case.


In [6]:
# Now apply it to A and B. Two rounds, printed as a table.
cA = [0] * 6
cB = [0] * 6
print(f"{'round':<7}{'colours of A':<22}{'multiset A':<18}{'colours of B':<22}{'multiset B'}")
print('-' * 92)
for r in range(3):
    print(f'{r:<7}{str(cA):<22}{str(sorted(cA)):<18}{str(cB):<22}{str(sorted(cB))}')
    if r < 2:
        cA, cB = wl_round(A_A, cA), wl_round(A_B, cB)

print()
print('colour multisets identical at every round? ',
      all(sorted(a) == sorted(b) for a, b in [(cA, cB)]))


round  colours of A          multiset A        colours of B          multiset B
--------------------------------------------------------------------------------------------
0      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]
1      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]
2      [0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0][0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 0, 0]

colour multisets identical at every round?  True


### YOUR ANSWER (Task 2)

Reproduce your **hand** calculation here: the colour of each atom in each graph at rounds
0, 1 and 2, and the colour multiset of each graph at each round. Confirm that it agrees
with the code output above.

*Write here:*



For Round 0, we initialize everything to color 0 since all nodes represent Carbon atoms. This gives both Graph A and Graph B a color array of [0, 0, 0, 0, 0, 0] and a multiset of {{0, 0, 0, 0, 0, 0}}.

Moving to Round 1, atom 0 in Graph A (the 6-ring) sees its two neighbors (atoms 1 and 5), resulting in a signature of (0, (0, 0)). This is identical for every other atom in the ring. Similarly, atom 0 in Graph B sees its two neighbors in the triangle (atoms 1 and 2), creating the exact same (0, (0, 0)) signature, which applies to all other atoms in Graph B as well. Because this is the only signature generated anywhere, it gets hashed back to color 0. The arrays and multisets remain [0, 0, 0, 0, 0, 0] and {{0, 0, 0, 0, 0, 0}}.

Round 2 just processes the same inputs as Round 1, so the output simply repeats as [0, 0, 0, 0, 0, 0] and {{0, 0, 0, 0, 0, 0}}. This perfectly aligns with our code output, which printed all 0s and a final True. Because the network's colors stabilized at Round 1, additional rounds are pointless, confirming that the 1-WL test will never be able to distinguish between Graph A and Graph B.

---
## Task 3. State the conclusion  &nbsp;&nbsp;

You now know what refinement does to A and B.


### YOUR ANSWER (Task 3)

Are A and B distinguishable by **any** message-passing model of the standard form

$$\bm{h}_i' = \phi_{\mathrm{upd}}\Big(\bm{h}_i,\ \bigoplus_{j \in \mathcal{N}(i)} \phi_{\mathrm{msg}}(\bm{h}_i, \bm{h}_j)\Big)?$$

Your justification must appeal to a **property of the two graphs**, not merely to the
outcome of your refinement. Two or three sentences.

Note carefully what the claim covers: it holds for every width, every depth, every choice
of $\phi$, and every amount of training data. Say why.

*Write here:*



Ultimately, these two graphs cannot be separated by standard message-passing architectures. Since both structures are strictly 2-regular with identical starting features, every node processes the exact same aggregated messages layer after layer. This creates a theoretical bottleneck: no matter how you adjust the model's width, depth, message functions, or training data, the network is guaranteed to output the exact same graph-level representation for both molecules.

---
## Task 4. Find a discriminating invariant  &nbsp;&nbsp;

Refinement failed. Something else must succeed, because the two graphs genuinely differ.

Recall the walk-counting theorem: $(A^k)_{ij}$ is the number of walks of length exactly
$k$ from atom $i$ to atom $j$. Therefore $\mathrm{tr}(A^k) = \sum_i (A^k)_{ii}$ counts
**closed** walks of length $k$, that is walks that return to where they started.


In [7]:
# TODO: compute the trace of the k-th matrix power for k = 2, 3 and 6.
#       Hint: np.linalg.matrix_power(A, k) and np.trace(...)

print(f"{'k':<5}{'tr(A_A^k)':>12}{'tr(A_B^k)':>12}   separates?")
print('-' * 45)
for k in [2, 3, 6]:
    tA = np.trace(np.linalg.matrix_power(A_A, k))      # <- replace
    tB = np.trace(np.linalg.matrix_power(A_B, k))     # <- replace
    print(f'{k:<5}{tA:>12.0f}{tB:>12.0f}   {"YES" if tA != tB else "no"}')


k       tr(A_A^k)   tr(A_B^k)   separates?
---------------------------------------------
2              12          12   no
3               0          12   YES
6             132         132   no


In [8]:
# Optional, not marked: the full adjacency spectra.
# For a conjugated system these are the Huckel orbital energies in units of beta.
print('eigenvalues of A_A:', np.sort(np.linalg.eigvalsh(A_A))[::-1])
print('eigenvalues of A_B:', np.sort(np.linalg.eigvalsh(A_B))[::-1])


eigenvalues of A_A: [ 2.  1.  1. -1. -1. -2.]
eigenvalues of A_B: [ 2.  2. -1. -1. -1. -1.]


### YOUR ANSWER (Task 4)

(a) Which value of $k$ separates A from B?

(b) Explain **in terms of walks** why that particular $k$ works. What closed walk exists
in one graph and not the other?

(c) Explain why the other two values of $k$ fail. Be careful with $k=6$: the result may
not be what you expected, and the explanation is the point of this part.

(d) One sentence: what does $\mathrm{tr}(A^2)$ count for *any* graph, and why could it
never have separated these two?

*Write here:*



(a) k = 3 successfully discriminates between the two.

(b) k = 3 detects triangles. Graph B contains 3-membered rings allowing 3-step closed walks, whereas Graph A lacks cycles smaller than 6.

(c) k = 2 fails because both structures have identical edge counts. k = 6 fails because the total number of 6-step closed walks (traversing the 6-ring in A vs. double-looping the 3-rings in B) coincidentally sum to the exact same number.

(d) Tr(A^2) calculates the sum of all node degrees (or twice the edge count); it fails here because both molecules are 2-regular networks with 6 bonds

---
## Task 5. Propose a fix and defend it  &nbsp;&nbsp;

Computing $\mathrm{tr}(A^k)$ costs $O(n^3)$ and does not transfer cleanly between
molecules of different size. A cheaper repair is to give each atom an extra **input
feature** that already distinguishes the two cases, so that the model separates them at
layer zero without any change to the architecture.

Implement your chosen feature below.


In [9]:
def extra_feature(A):
    """Return a length-n array: one extra scalar feature per atom.

    TODO: choose a feature that (i) differs between A and B,
          (ii) is computable in linear time by any cheminformatics toolkit,
          (iii) is chemically meaningful, not an arbitrary code.
    """
    n = A.shape[0]
    E = int(A.sum() // 2)
    visited = [False] * n
    C = 0

    for start in range(n):
        if not visited[start]:
            C += 1

            stack = [start]
            visited[start] = True

            while stack:
                i = stack.pop()

                for j in range(n):
                    if A[i, j] == 1 and not visited[j]:
                        visited[j] = True
                        stack.append(j)
    feature = np.zeros(n)
    ring_count = E - n + C
    feature = np.full(n, ring_count, dtype=float)
    # TODO: fill in
    return feature

fA, fB = extra_feature(A_A), extra_feature(A_B)
print('feature on A:', fA)
print('feature on B:', fB)


feature on A: [1. 1. 1. 1. 1. 1.]
feature on B: [2. 2. 2. 2. 2. 2.]


In [10]:
# CHECK: does the feature actually do the job?
assert not np.allclose(np.sort(fA), np.sort(fB)), (
    'Your feature takes the same multiset of values on A and B, '
    'so it cannot separate them. Try again.')
print('The feature separates A from B at the input layer.')

# and does it survive relabelling of the atoms?
perm = np.random.default_rng(0).permutation(6)
A_A_perm = A_A[np.ix_(perm, perm)]
assert np.allclose(np.sort(extra_feature(A_A_perm)), np.sort(fA)), (
    'Your feature changes when the atoms are relabelled. It must be permutation equivariant.')
print('The feature is unchanged by relabelling the atoms, as required.')


The feature separates A from B at the input layer.
The feature is unchanged by relabelling the atoms, as required.


### YOUR ANSWER (Task 5)

(a) Name your feature and state its value on every atom of A and of B.

(b) Justify why it is cheap: what algorithm computes it, and at what cost?

(c) Name **one further chemical property** that this feature would help a model predict,
and say why.

(d) One sentence on the general lesson: when a model provably cannot see something, is
the better response a bigger architecture or a better input feature?

*Write here:*



(a) Feature & Values: The total number of independent rings (cyclomatic number). Every atom in Graph A is assigned a value of 1 (one 6-membered ring), while every atom in Graph B is assigned a value of 2 (two 3-membered rings).

(b) Algorithm & Cost: This computes in O(V + E) time, making it highly efficient. By using a standard DFS or BFS to find connected components (C), you can simply apply the cyclomatic formula (E - V + C). For sparse molecular graphs, this scales linearly.

(c) Chemical Property: Ring strain. Explicitly passing ring topology helps the model assess molecular tension, as smaller rings (e.g., cyclopropanes) suffer from significantly higher angular strain than larger ones.

(d) Key Lesson: When an architecture has a mathematical blind spot, engineering a chemically meaningful input feature is a far smarter and cheaper fix than simply increasing the model's size

---
## Before you submit

Run the cell below. It confirms only that the notebook executes; it does not mark your
prose answers.


In [11]:
checks = {
    'Task 1: adjacency matrices built': (A_A.sum() == 12 and A_B.sum() == 12
                                          and not np.allclose(A_A, A_B)),
    'Task 2: wl_round implemented':      sorted(wl_round(A_nbutane, [0]*4)) != sorted(wl_round(A_isobutane, [0]*4)),
    'Task 5: extra_feature implemented': not np.allclose(extra_feature(A_A), 0),
}
for k, v in checks.items():
    print(('  OK   ' if v else '  TODO ') + k)
print()
print('Remember: Kernel > Restart & Run All, then save with all output visible.')


  OK   Task 1: adjacency matrices built
  OK   Task 2: wl_round implemented
  OK   Task 5: extra_feature implemented

Remember: Kernel > Restart & Run All, then save with all output visible.
